# Walk-Forward ML Filter

This notebook demonstrates the leakage-aware modeling workflow used by the repository. The classifier is trained only on past candidate trades and predicts whether the baseline signal should be accepted.

## Validation Design

The workflow uses chronological splits and rolling windows. Scaling is fitted on each training window only. Threshold selection uses only training-window probabilities and historical PnL, then the selected threshold is applied to the next test window.

In [ ]:
from sklearn.linear_model import LogisticRegression

from src.data import load_synchronized_sample
from src.features import add_market_features, feature_columns
from src.labels import generate_labeled_trades
from src.validation import chronological_split, run_walk_forward_classifier

market = add_market_features(load_synchronized_sample())
trades = generate_labeled_trades(market, max_holding_minutes=60)
cols = feature_columns(trades.columns)
cols

In [ ]:
split = chronological_split(trades, train_frac=0.7, validation_frac=0.0)
len(split.train), len(split.test)

In [ ]:
result = run_walk_forward_classifier(
    split.train,
    cols,
    model_factory=lambda: LogisticRegression(
        solver="liblinear",
        C=0.1,
        class_weight="balanced",
        random_state=42,
    ),
    train_window=40,
    test_window=20,
    threshold_grid=[0.50, 0.55, 0.60],
)
result["metrics"]

In [ ]:
result["selected_trades"][["entry_time", "pnl", "probability", "signal"]].head()

## Reading the Result

The sample workflow is a reproducibility check, not an investment claim. The full local archive contained broader experiments, but this public version keeps claims constrained to what can be audited: point-in-time features, no target leakage, chronological validation, and explicit limitations.